# EXPORT DOCS MANAGEMENT — chạy trên Google Colab

Quản lý bộ chứng từ xuất khẩu container, chạy ngay trên trình duyệt, **dữ liệu lưu trong Google Drive của bạn**.

**Cách dùng:** chạy lần lượt từng ô **1 → 6** (bấm nút ▶ bên trái mỗi ô, chờ ô xong rồi mới chạy ô tiếp theo).
Ô 6 in ra **đường link** để mở ứng dụng. Hướng dẫn đầy đủ: file **COLAB_GUIDE.md**.

Trước lần đầu: tải file `ExportDocs_app.zip` lên thư mục **`MyDrive/ExportDocs_app`** trên Google Drive.

- Dữ liệu (lô hàng, PDF, file tải lên, bản sao lưu): `MyDrive/ExportDocs_data` — **không mất khi Colab dừng**.
- Mỗi lần mở lại: chạy lại các ô 1 → 6, các lô hàng cũ vẫn còn nguyên.

## 1️⃣ Cài thư viện
Cài các thư viện ứng dụng cần (Gradio, ReportLab, …), **đúng phiên bản đã kiểm tra**. Mất khoảng 1–2 phút.
Nếu pip in ra chữ đỏ *“dependency conflicts”* thì không sao — đó là các gói khác của Colab, ứng dụng không dùng.

In [ ]:
# @title 1️⃣ Cài thư viện (khoảng 1–2 phút)
# Cài đúng phiên bản thư viện đã kiểm tra (giống requirements.txt trong file ZIP).
import importlib
import importlib.metadata as md
import subprocess
import sys

GOI_CAN_CAI = [
    "gradio==6.32.0",
    "reportlab==5.1.0",
    "qrcode==8.2",
    "pypdfium2==5.15.2",
    "pandas==3.1.1 ; python_version >= \"3.14\"",
    "pandas==2.2.3 ; python_version < \"3.14\"",
]

print(f"Python trên Colab: {sys.version.split()[0]}")
if sys.version_info < (3, 11):
    raise SystemExit("⛔ Ứng dụng cần Python 3.11 trở lên. Chọn Runtime → Change runtime type → phiên bản mới nhất.")

# Các gói Colab có thể đã nạp sẵn: nếu pip đổi phiên bản của chúng, Colab cần khởi động lại một lần.
_DA_NAP = {"numpy": "numpy", "pandas": "pandas", "PIL": "pillow", "pydantic": "pydantic", "fastapi": "fastapi",
           "starlette": "starlette", "httpx": "httpx", "anyio": "anyio", "typing_extensions": "typing_extensions",
           "yaml": "PyYAML", "jinja2": "Jinja2", "huggingface_hub": "huggingface-hub", "gradio": "gradio"}


def _phien_ban(goi):
    try:
        return md.version(goi)
    except md.PackageNotFoundError:
        return None


_truoc = {goi: _phien_ban(goi) for mod, goi in _DA_NAP.items() if mod in sys.modules}
_kq = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *GOI_CAN_CAI],
                     capture_output=True, text=True)
if _kq.returncode != 0:
    print(_kq.stdout[-3000:], _kq.stderr[-3000:], sep="\n")
    raise SystemExit("⛔ Cài thư viện không thành công — xem lỗi phía trên rồi chạy lại ô này.")
importlib.invalidate_caches()
for goi in ("gradio", "reportlab", "pandas", "pypdfium2", "qrcode"):
    print(f"  ✓ {goi} {_phien_ban(goi)}")
_doi = [goi for goi, v in _truoc.items() if _phien_ban(goi) != v]
if _doi:
    print("\n⚠️ Colab cần KHỞI ĐỘNG LẠI một lần (vì đã đổi phiên bản: " + ", ".join(_doi) + ").\n"
          "   Bấm menu Runtime → Restart session, rồi chạy lại từ ô 1 (lần này rất nhanh).")
else:
    print("\n✅ Đã cài xong. Chạy tiếp ô 2.")

## 2️⃣ Kết nối Google Drive và tạo thư mục dữ liệu
Colab sẽ hỏi quyền truy cập Google Drive: chọn **tài khoản Google của bạn** rồi bấm **Cho phép / Allow**.
Dữ liệu được lưu trong `MyDrive/ExportDocs_data` (tự tạo nếu chưa có). Đây là nơi giữ dữ liệu lâu dài.

In [ ]:
# @title 2️⃣ Kết nối Google Drive và tạo thư mục dữ liệu
# Dữ liệu lâu dài nằm trên Drive. Colab chỉ giữ một bản sao làm việc (tự chép lên Drive sau mỗi lần lưu).
import os
import time
from datetime import datetime
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

THU_MUC_DU_LIEU = "/content/drive/MyDrive/ExportDocs_data"  # @param {type:"string"}
THU_MUC_LAM_VIEC = "/content/ExportDocs_work"   # ổ đĩa tạm của Colab — mất khi Colab dừng, KHÔNG sao

Path(THU_MUC_DU_LIEU).mkdir(parents=True, exist_ok=True)
os.environ["EXPORT_DOCS_DATA_DIR"] = THU_MUC_DU_LIEU
os.environ["EXPORT_DOCS_LOCAL_DB"] = f"{THU_MUC_LAM_VIEC}/export_docs.sqlite3"
os.environ["EXPORT_DOCS_TIMEZONE"] = "Asia/Ho_Chi_Minh"     # giờ Việt Nam cho hạn cut-off
os.environ["TZ"] = "Asia/Ho_Chi_Minh"
time.tzset()

_db = Path(THU_MUC_DU_LIEU) / "database" / "export_docs.sqlite3"
print(f"📁 Thư mục dữ liệu: {THU_MUC_DU_LIEU}")
if _db.exists():
    _luc = datetime.fromtimestamp(_db.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
    print(f"✅ Đã có dữ liệu trên Drive ({_db.stat().st_size / 1024:,.0f} KB, lưu lần cuối {_luc}).")
else:
    print("ℹ️ Chưa có dữ liệu — lần chạy đầu tiên sẽ tạo mới.")
if (Path(THU_MUC_DU_LIEU) / "ExportDocs_DANG_MO.lock").exists():
    print("ℹ️ Có file khóa ExportDocs_DANG_MO.lock (dữ liệu đang / vừa được mở) — ô 4 sẽ kiểm tra.")
print("Chạy tiếp ô 3.")

## 3️⃣ Lấy mã ứng dụng
Mở file **`ExportDocs_app.zip`** trong thư mục `MyDrive/ExportDocs_app` trên Drive (bạn tải lên một lần) và giải nén
vào Colab. Bên trong có sẵn **phông chữ tiếng Việt** (Noto Sans) cho PDF. Nếu có nhiều file ZIP, ô này dùng bản có
**phiên bản mới nhất**. Cập nhật ứng dụng = thay file ZIP trên Drive; dữ liệu không bị đụng tới.

In [ ]:
# @title 3️⃣ Lấy mã ứng dụng (file ZIP trên Google Drive)
import os
import shutil
import sys
import zipfile
from pathlib import Path

THU_MUC_MA = "/content/drive/MyDrive/ExportDocs_app"  # @param {type:"string"}
NOI_GIAI_NEN = "/content/ExportDocs_app"


def _phien_ban_zip(file_zip):
    with zipfile.ZipFile(file_zip) as z:
        return z.read("ExportDocs_app/VERSION").decode("utf-8").strip()


def _so(v):
    return tuple(int(x) if x.isdigit() else 0 for x in v.split("."))


if "export_docs" in sys.modules:
    import export_docs
    print(f"ℹ️ Mã ứng dụng đã được nạp trong phiên này. Phiên bản ứng dụng: {export_docs.__version__}\n"
          "   Muốn dùng file ZIP mới: menu Runtime → Restart session, rồi chạy lại từ ô 1.")
else:
    _cac_file = sorted(Path(THU_MUC_MA).glob("ExportDocs_app*.zip"))
    if not _cac_file:
        raise SystemExit(f"⛔ Không tìm thấy file ExportDocs_app.zip trong {THU_MUC_MA}.\n"
                         "   Hãy tải file ZIP lên đúng thư mục này trên Google Drive (COLAB_GUIDE.md, Bước 1), "
                         "rồi chạy lại ô này.")
    _ung_vien = []
    for _f in _cac_file:
        try:
            _v = _phien_ban_zip(_f)
            _ung_vien.append((_so(_v), _v, str(_f)))
        except (KeyError, zipfile.BadZipFile):
            print(f"⚠️ Bỏ qua {_f.name}: không phải file ZIP của ứng dụng.")
    if not _ung_vien:
        raise SystemExit("⛔ Không có file ZIP hợp lệ của ứng dụng trong thư mục.")
    _, PHIEN_BAN, _zip = max(_ung_vien)
    if len(_ung_vien) > 1:
        print(f"ℹ️ Có {len(_ung_vien)} file ZIP — dùng bản mới nhất: {Path(_zip).name}")
    shutil.rmtree(NOI_GIAI_NEN, ignore_errors=True)
    with zipfile.ZipFile(_zip) as z:
        for _ten in z.namelist():
            if not _ten.startswith("ExportDocs_app/") or ".." in _ten.split("/"):
                raise SystemExit(f"⛔ File ZIP không hợp lệ ({_ten}).")
        z.extractall("/content")
    _fonts = sorted(p.name for p in (Path(NOI_GIAI_NEN) / "export_docs" / "assets" / "fonts").glob("*.ttf"))
    if len(_fonts) < 3:
        raise SystemExit("⛔ Thiếu phông chữ tiếng Việt trong file ZIP — hãy tạo lại file ZIP.")
    _goi_zip = [l.split("#", 1)[0].strip() for l in
                (Path(NOI_GIAI_NEN) / "requirements.txt").read_text(encoding="utf-8").splitlines()]
    if "GOI_CAN_CAI" in globals() and sorted(filter(None, _goi_zip)) != sorted(GOI_CAN_CAI):
        print("⚠️ File ZIP này cần phiên bản thư viện khác với ô 1 của notebook. Hãy dùng notebook đi kèm file ZIP.")
    sys.path.insert(0, NOI_GIAI_NEN)
    os.chdir(NOI_GIAI_NEN)
    print(f"✅ Đã giải nén {Path(_zip).name}. Phông chữ: {', '.join(_fonts)}")
    print(f"Phiên bản ứng dụng: {PHIEN_BAN}")
    print("Chạy tiếp ô 4.")

## 4️⃣ Mở dữ liệu (khởi tạo cơ sở dữ liệu)
- Đặt **khóa** trong thư mục dữ liệu, để một phiên Colab khác không mở cùng dữ liệu một lúc
  (nếu dữ liệu đang mở ở nơi khác, ô này báo **cảnh báo bằng tiếng Việt** và dừng lại).
- Chép cơ sở dữ liệu từ Drive về Colab, tạo / nâng cấp các bảng (dữ liệu cũ giữ nguyên).
- Chỉ bật **MO_KHOA_CU** khi chắc chắn không còn phiên Colab nào khác đang mở dữ liệu.

In [ ]:
# @title 4️⃣ Mở dữ liệu (khóa, chép từ Drive, tạo bảng)
MO_KHOA_CU = False  # @param {type:"boolean"}

if "THU_MUC_DU_LIEU" not in globals() or "NOI_GIAI_NEN" not in globals():
    raise SystemExit("⛔ Hãy chạy ô 2 và ô 3 trước.")

import app
from export_docs import database, drive_sync

try:
    app.prepare_data(force_unlock=MO_KHOA_CU)
except drive_sync.DataLockedError as _loi:
    print(_loi)
    raise SystemExit("⛔ Dừng lại — dữ liệu đang được mở ở phiên Colab khác (xem hướng dẫn phía trên).") from None

with database.reading() as _conn:
    _so_lo = _conn.execute("SELECT COUNT(*) FROM shipments").fetchone()[0]
print(f"\n✅ Dữ liệu sẵn sàng: {_so_lo} lô hàng.")
print("   Chạy tiếp ô 5 (tùy chọn, dữ liệu mẫu) hoặc ô 6 (mở ứng dụng).")

## 5️⃣ (Tùy chọn) Lô hàng MẪU
Mặc định **TẮT**. Bật `TAI_DU_LIEU_MAU` rồi chạy ô này nếu muốn có lô hàng mẫu **SHP-2026-0001**
(đánh dấu **SAMPLE / DEMO**, số liệu giả để tập dùng — không phải chứng từ thật). Chỉ tạo một lần.

In [ ]:
# @title 5️⃣ (Tùy chọn) Tải lô hàng MẪU SHP-2026-0001 — mặc định TẮT
TAI_DU_LIEU_MAU = False  # @param {type:"boolean"}

if "app" not in globals():
    raise SystemExit("⛔ Hãy chạy ô 4 trước.")
if TAI_DU_LIEU_MAU:
    app.load_demo()
    print("✅ Đã có lô hàng mẫu SHP-2026-0001 (SAMPLE / DEMO — dữ liệu giả, chỉ để tập dùng).")
else:
    print("Bỏ qua dữ liệu mẫu (TAI_DU_LIEU_MAU = False). Chạy tiếp ô 6.")

## 6️⃣ Mở ứng dụng
In ra **đường link công khai** (dạng `https://….gradio.live`) — bấm vào để mở ứng dụng, dùng được trên máy tính và điện thoại.
- **Ai có link đều mở được ứng dụng** (không có đăng nhập) — đừng gửi link cho người ngoài.
- Link chỉ hoạt động **khi ô này đang chạy** và **hết hạn sau một thời gian**.
- Muốn dừng: bấm nút **■** của ô này (dữ liệu được chép lên Drive lần cuối và khóa được gỡ).
- Link không vào được nữa? Bấm **■** rồi chạy lại ô 6. Nếu Colab đã ngắt kết nối: chạy lại các ô 1 → 6.
  Dữ liệu vẫn an toàn trên Drive.

In [ ]:
# @title 6️⃣ Mở ứng dụng — in đường link công khai
import time

from IPython.display import HTML, display

if "app" not in globals():
    raise SystemExit("⛔ Hãy chạy ô 4 trước.")
from export_docs import drive_sync

if "demo" in globals():
    try:
        demo.close()        # chạy lại ô này: đóng ứng dụng cũ trước
    except Exception:
        pass
if not drive_sync.status()["started"]:      # đã dừng bằng ■ trước đó → khóa và chép từ Drive lại
    try:
        app.prepare_data()
    except drive_sync.DataLockedError as _loi:
        print(_loi)
        raise SystemExit("⛔ Dừng lại — dữ liệu đang được mở ở phiên Colab khác.") from None

demo = app.build()
demo.launch(share=True, inline=False, prevent_thread_lock=True, **app.launch_options())
LINK = demo.share_url

if not LINK:
    demo.close()
    raise SystemExit("⛔ Chưa tạo được link công khai (mạng Gradio đang bận). Chờ một chút rồi chạy lại ô 6.")

print("=" * 72)
print("✅ ỨNG DỤNG ĐANG CHẠY. Mở đường link này (bấm vào, hoặc sao chép):")
print()
print("    " + LINK)
print()
print("• Link chỉ hoạt động khi ô này đang chạy, và hết hạn sau một thời gian.")
print("• Đừng chia sẻ link cho người ngoài: ai có link đều mở được ứng dụng và dữ liệu.")
print("• Muốn dừng: bấm nút ■ của ô này.")
print("• Link không vào được? Bấm ■ rồi chạy lại ô 6. Colab đã ngắt kết nối? Chạy lại ô 1 → 6.")
print("=" * 72)
display(HTML(f'<p style="font-size:18px">👉 <a href="{LINK}" target="_blank"><b>Mở EXPORT DOCS MANAGEMENT</b></a></p>'))

_loi_cu = None
try:
    while True:                      # giữ ô chạy; báo nếu không chép được lên Drive
        time.sleep(10)
        _loi = drive_sync.status()["last_error"]
        if _loi and _loi != _loi_cu:
            print("⚠️ " + _loi)
        _loi_cu = _loi
except KeyboardInterrupt:
    print("Đang dừng ứng dụng …")
finally:
    demo.close()
    app.shutdown()
    print(f"✅ Đã dừng. Dữ liệu đã được chép lên Drive ({THU_MUC_DU_LIEU}) và khóa đã được gỡ.")